In [2]:
import pandas as pd
from pathlib import Path

## Clean Data

In [19]:
input_data = Path("Zomato Restaurant Dataset.csv")

# Load the dataset
df_raw = pd.read_csv(input_data, encoding="utf-8-sig")
df = df_raw.copy()
print("Successfully loaded the dataset with shape:", df.shape)

Successfully loaded the dataset with shape: (9551, 21)


In [7]:
# Normalize the column names
df.columns = df.columns.str.strip()
print(df.columns)

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='str')


In [18]:
initial_rows = len(df)

# Data-quality flags
df["Missing Cuisine"] = df["Cuisines"].isna()
df["Is Unrated"] = df["Aggregate rating"].eq(0)
df["Has Zero Cost"] = df["Average Cost for two"].eq(0)
df["Has Zero Coordinates"] = df[["Latitude", "Longitude"]].eq(0).any(axis=1)

assert len(df) == initial_rows, "Row count changed after adding flags"
assert df["Restaurant ID"].is_unique, "Restaurant ID is not unique"
assert df["Aggregate rating"].between(0, 5).all(), "Aggregate rating is out of bounds"

# Printing summary statistics
print("Initial rows:", initial_rows)
print("Final rows:", len(df))
print("Missing Cuisine:", df["Missing Cuisine"].sum())
print("Unrated restaurants:", df["Is Unrated"].sum())
print("Restaurants with zero cost:", df["Has Zero Cost"].sum())
print("Restaurants with zero coordinates:", df["Has Zero Coordinates"].sum())

Initial rows: 9551
Final rows: 9551
Missing Cuisine: 9
Unrated restaurants: 2148
Restaurants with zero cost: 18
Restaurants with zero coordinates: 499


# Restaurant Distribution

## Country Distribution

In [25]:
country_summary = (
    df.groupby(["Country Code", "Currency"]).size().reset_index(name="Restaurant Count").sort_values(by="Restaurant Count", ascending=False)
)
country_summary["Share (%)"] = (
    round(country_summary["Restaurant Count"] / country_summary["Restaurant Count"].sum() * 100, 2)
)
print(country_summary)

    Country Code                Currency  Restaurant Count  Share (%)
0              1      Indian Rupees(Rs.)              8652      90.59
14           216               Dollar($)               434       4.54
13           215              Pounds(��)                80       0.84
12           214      Emirati Diram(AED)                60       0.63
9            189                 Rand(R)                60       0.63
2             30      Brazilian Real(R$)                60       0.63
5            148           NewZealand($)                40       0.42
11           208        Turkish Lira(TL)                34       0.36
1             14               Dollar($)                24       0.25
6            162        Botswana Pula(P)                22       0.23
4             94  Indonesian Rupiah(IDR)                21       0.22
7            166         Qatari Rial(QR)                20       0.21
10           191   Sri Lankan Rupee(LKR)                20       0.21
8            184    

## Cities Distribution

In [29]:
city_summary = (
    df.groupby(["Country Code", "City"]).size().reset_index(name="Restaurant Count").sort_values(by="Restaurant Count", ascending=False)
)
city_summary["Share (%)"] = (
    round(city_summary["Restaurant Count"] / city_summary["Restaurant Count"].sum() * 100, 2)
)
city_summary.head(10)

,Country Code,City,Restaurant Count,Share (%)
31,1,New Delhi,5473,57.30
15,1,Gurgaon,1118,11.71
32,1,Noida,1080,11.31
12,1,Faridabad,251,2.63
13,1,Ghaziabad,25,0.26
3,1,Amritsar,21,0.22
1,1,Ahmedabad,21,0.22
23,1,Lucknow,21,0.22
16,1,Guwahati,21,0.22
7,1,Bhubaneshwar,21,0.22


**Conclusion**
- As we can see, the restaurants are heavily concentrated in India with over 90% of restaurants built there. This is reasonable as Zomato is an Indian brand. The 2nd largest market is US with around 4.5%.
- Furthermore, when we look at it from the cities, it's even more concentrated within New Dehli, over 57% of all restaurants situated here.

=> Further analysis, especially rating analysis will possibly be heavily skewed by this region preference.

# Restaurant Ratings